# ⚡️ Energy-Based Models

In this notebook, we'll walk through the steps required to train your own Energy Based Model to predict the distribution of a demo dataset

The code is adapted from the excellent ['Deep Energy-Based Generative Models' tutorial](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial8/Deep_Energy_Models.html) created by Phillip Lippe.

In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import (
    datasets,
    layers,
    models,
    optimizers,
    activations,
    metrics,
    callbacks,
)
import random
import matplotlib.pyplot as plt

try:
    from notebooks.utils import display, sample_batch
except ImportError:
    def display(imgs, save_to=None):
        n = min(len(imgs), 10)
        plt.figure(figsize=(20, 2))
        for i in range(n):
            plt.subplot(1, n, i + 1)
            plt.imshow(imgs[i].reshape(32, 32), cmap="gray")
            plt.axis("off")
        if save_to: plt.savefig(save_to)
        plt.show()
    def sample_batch(dataset):
        return next(iter(dataset))[:10]

## 0. Parameters <a name="parameters"></a>

In [1]:
IMAGE_SIZE = 32
CHANNELS = 1
STEP_SIZE = 10
STEPS = 60
NOISE = 0.005
ALPHA = 0.1
GRADIENT_CLIP = 0.03
BATCH_SIZE = 128
BUFFER_SIZE = 8192
LEARNING_RATE = 0.0001
EPOCHS = 120
LOAD_MODEL = False

In [23]:
(x_train, _), (x_test, _) = datasets.mnist.load_data()

def preprocess(imgs):
    imgs = (imgs.astype("float32") - 127.5) / 127.5
    imgs = np.pad(imgs, ((0, 0), (2, 2), (2, 2)), constant_values=-1.0)
    imgs = np.expand_dims(imgs, -1)
    return imgs

x_train = preprocess(x_train)
x_test = preprocess(x_test)

x_train = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
x_test = tf.data.Dataset.from_tensor_slices(x_test).batch(BATCH_SIZE)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [14]:
# Preprocess the data
def preprocess(imgs):
    imgs = (imgs.astype("float32") - 127.5) / 127.5
    imgs = np.pad(imgs, ((0, 0), (2, 2), (2, 2)), constant_values=-1.0)
    imgs = np.expand_dims(imgs, -1)
    return imgs

x_train = preprocess(x_train)
x_test = preprocess(x_test)

NameError: name 'x_train' is not defined

In [15]:
x_train = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
x_test = tf.data.Dataset.from_tensor_slices(x_test).batch(BATCH_SIZE)

NameError: name 'tf' is not defined

In [ ]:
# Show some items of clothing from the training set
train_sample = sample_batch(x_train)
display(train_sample)

## 2. Build the EBM network <a name="train"></a>

In [24]:
ebm_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
x = layers.Conv2D(16, kernel_size=5, strides=2, padding="same", activation=activations.swish)(ebm_input)
x = layers.Conv2D(32, kernel_size=3, strides=2, padding="same", activation=activations.swish)(x)
x = layers.Conv2D(64, kernel_size=3, strides=2, padding="same", activation=activations.swish)(x)
x = layers.Conv2D(64, kernel_size=3, strides=2, padding="same", activation=activations.swish)(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation=activations.swish)(x)
ebm_output = layers.Dense(1)(x)
model = models.Model(ebm_input, ebm_output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 16, 16, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 32)       │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 2, 2, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,993 (300.75 KB)

 Trainable params: 76,993 (300.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
if LOAD_MODEL:
    model.load_weights("./models/model.h5")

## 2. Set up a Langevin sampler function <a name="sampler"></a>

In [25]:
def generate_samples(model, inp_imgs, steps, step_size, noise, return_img_per_step=False):
    imgs_per_step = []
    for _ in range(steps):
        inp_imgs += tf.random.normal(inp_imgs.shape, mean=0, stddev=noise)
        inp_imgs = tf.clip_by_value(inp_imgs, -1.0, 1.0)
        with tf.GradientTape() as tape:
            tape.watch(inp_imgs)
            out_score = model(inp_imgs)
        grads = tape.gradient(out_score, inp_imgs)
        grads = tf.clip_by_value(grads, -GRADIENT_CLIP, GRADIENT_CLIP)
        inp_imgs += step_size * grads
        inp_imgs = tf.clip_by_value(inp_imgs, -1.0, 1.0)
        if return_img_per_step: imgs_per_step.append(inp_imgs)
    return tf.stack(imgs_per_step, axis=0) if return_img_per_step else inp_imgs

class Buffer:
    def __init__(self, model):
        self.model = model
        self.examples = [tf.random.uniform(shape=(1, 32, 32, 1)) * 2 - 1 for _ in range(BATCH_SIZE)]
    def sample_new_exmps(self, steps, step_size, noise):
        n_new = np.random.binomial(BATCH_SIZE, 0.05)
        rand_imgs = tf.random.uniform((n_new, 32, 32, 1)) * 2 - 1
        old_imgs = tf.concat(random.choices(self.examples, k=BATCH_SIZE - n_new), axis=0)
        inp_imgs = tf.concat([rand_imgs, old_imgs], axis=0)
        inp_imgs = generate_samples(self.model, inp_imgs, steps=steps, step_size=step_size, noise=noise)
        self.examples = tf.split(inp_imgs, BATCH_SIZE, axis=0) + self.examples
        self.examples = self.examples[:BUFFER_SIZE]
        return inp_imgs

## 3. Set up a buffer to store examples <a name="buffer"></a>

In [11]:
class Buffer:
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.examples = [
            tf.random.uniform(shape=(1, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)) * 2
            - 1
            for _ in range(BATCH_SIZE)
        ]

    def sample_new_exmps(self, steps, step_size, noise):
        n_new = np.random.binomial(BATCH_SIZE, 0.05)
        rand_imgs = (
            tf.random.uniform((n_new, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)) * 2 - 1
        )
        old_imgs = tf.concat(
            random.choices(self.examples, k=BATCH_SIZE - n_new), axis=0
        )
        inp_imgs = tf.concat([rand_imgs, old_imgs], axis=0)
        inp_imgs = generate_samples(
            self.model, inp_imgs, steps=steps, step_size=step_size, noise=noise
        )
        self.examples = tf.split(inp_imgs, BATCH_SIZE, axis=0) + self.examples
        self.examples = self.examples[:BUFFER_SIZE]
        return inp_imgs

In [32]:
class EBM(models.Model):
    def __init__(self, energy_model):
        super(EBM, self).__init__()
        self.model = energy_model
        self.buffer = Buffer(self.model)
        self.alpha = ALPHA
        self.loss_metric = metrics.Mean(name="loss")
        self.reg_loss_metric = metrics.Mean(name="reg")
        self.cdiv_loss_metric = metrics.Mean(name="cdiv")
        self.real_out_metric = metrics.Mean(name="real")
        self.fake_out_metric = metrics.Mean(name="fake")

    @property
    def metrics(self):
        return [self.loss_metric, self.reg_loss_metric, self.cdiv_loss_metric, self.real_out_metric, self.fake_out_metric]

    def call(self, x):
        return self.model(x)

    def train_step(self, real_imgs):
        real_imgs += tf.random.normal(shape=tf.shape(real_imgs), mean=0, stddev=NOISE)
        real_imgs = tf.clip_by_value(real_imgs, -1.0, 1.0)
        fake_imgs = self.buffer.sample_new_exmps(steps=STEPS, step_size=STEP_SIZE, noise=NOISE)
        inp_imgs = tf.concat([real_imgs, fake_imgs], axis=0)
        with tf.GradientTape() as training_tape:
            real_out, fake_out = tf.split(self.model(inp_imgs), 2, axis=0)
            cdiv_loss = tf.reduce_mean(fake_out) - tf.reduce_mean(real_out)
            reg_loss = self.alpha * tf.reduce_mean(real_out**2 + fake_out**2)
            loss = cdiv_loss + reg_loss
        grads = training_tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        self.loss_metric.update_state(loss)
        self.reg_loss_metric.update_state(reg_loss)
        self.cdiv_loss_metric.update_state(cdiv_loss)
        self.real_out_metric.update_state(tf.reduce_mean(real_out))
        self.fake_out_metric.update_state(tf.reduce_mean(fake_out))
        return {m.name: m.result() for m in self.metrics}

In [37]:
ebm = EBM(model)
# We compile without a standard loss because the loss is calculated inside train_step
ebm.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    run_eagerly=True
)
# Warmup: Initialize weights by calling the model once
ebm(np.zeros((1, IMAGE_SIZE, IMAGE_SIZE, CHANNELS), dtype=np.float32))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-1.9972188]], dtype=float32)>

## 3. Train the EBM network <a name="train"></a>

In [8]:
# Compile the model
ebm.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), run_eagerly=True
)

NameError: name 'ebm' is not defined

In [27]:
import os
if not os.path.exists("./output"): os.makedirs("./output")

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

class ImageGenerator(callbacks.Callback):
    def __init__(self, num_img):
        self.num_img = num_img
    def on_epoch_end(self, epoch, logs=None):
        start_imgs = np.random.uniform(size=(self.num_img, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)) * 2 - 1
        generated_images = generate_samples(self.model.model, start_imgs, steps=1000, step_size=STEP_SIZE, noise=NOISE, return_img_per_step=False)
        display(generated_images.numpy(), save_to="./output/generated_img_%03d.png" % (epoch))

image_generator_callback = ImageGenerator(num_img=10)

In [28]:
if not os.path.exists("./models"): os.makedirs("./models")

class SaveModel(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.model.save_weights("./models/model.h5")

save_model_callback = SaveModel()

In [38]:
ebm.fit(x_train, epochs=120, validation_data=x_test, callbacks=[save_model_callback, tensorboard_callback, image_generator_callback])

Epoch 1/120
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - cdiv: -0.0252 - fake: 0.0018 - loss: -0.0217 - real: 0.0270 - reg: 0.0036

ValueError: No loss to compute. Provide a `loss` argument in `compile()`.

In [36]:
import matplotlib.pyplot as plt

def plot_ebm_metrics(history):
    plt.figure(figsize=(12, 5))

    # Plot CDiv and Total Loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history.get('loss', []), label='Total Loss')
    plt.plot(history.history.get('cdiv', []), label='Contrastive Divergence')
    plt.title('EBM Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)

    # Plot Real vs Fake scores
    plt.subplot(1, 2, 2)
    plt.plot(history.history.get('real', []), label='Real Energy')
    plt.plot(history.history.get('fake', []), label='Fake Energy')
    plt.title('Energy Scores (Lower is higher probability)')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

if 'ebm' in globals() and hasattr(ebm, 'history') and len(ebm.history.history.get('loss', [])) > 0:
    plot_ebm_metrics(ebm.history)
else:
    print('Training in progress (Epoch {}/{}). Please wait for ebm.fit to finish to see the curve.'.format(len(ebm.history.history.get('loss', [])) if hasattr(ebm, 'history') else 0, EPOCHS))

Training in progress (Epoch 0/120). Please wait for ebm.fit to finish to see the curve.


In [35]:
import matplotlib.pyplot as plt

def plot_ebm_metrics(history):
    plt.figure(figsize=(12, 5))

    # Plot CDiv and Total Loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history.get('loss', []), label='Total Loss')
    plt.plot(history.history.get('cdiv', []), label='Contrastive Divergence')
    plt.title('EBM Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)

    # Plot Real vs Fake scores
    plt.subplot(1, 2, 2)
    plt.plot(history.history.get('real', []), label='Real Energy')
    plt.plot(history.history.get('fake', []), label='Fake Energy')
    plt.title('Energy Scores (Lower is higher probability)')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

if 'ebm' in globals() and hasattr(ebm, 'history') and len(ebm.history.history.get('loss', [])) > 0:
    plot_ebm_metrics(ebm.history)
else:
    print('Training in progress (Epoch {}/{}). Please wait for ebm.fit to finish to see the curve.'.format(len(ebm.history.history.get('loss', [])) if hasattr(ebm, 'history') else 0, EPOCHS))

Training in progress (Epoch 0/120). Please wait for ebm.fit to finish to see the curve.


In [31]:
import matplotlib.pyplot as plt

def plot_loss(history):
    plt.figure(figsize=(10, 6))
    # Using get() to handle cases where metrics might have prefixes
    loss = history.history.get('loss')
    cdiv = history.history.get('cdiv')
    reg = history.history.get('reg')

    if loss:
        plt.plot(loss, label='Total Loss')
    if cdiv:
        plt.plot(cdiv, label='Contrastive Divergence')
    if reg:
        plt.plot(reg, label='Regularization')

    plt.title('EBM Training Loss Curve (120 Epochs)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.grid(True)
    plt.show()

# Ensure training is complete before plotting
if 'ebm' in globals() and hasattr(ebm, 'history') and len(ebm.history.history.get('loss', [])) > 0:
    plot_loss(ebm.history)
else:
    print('Training in progress or history empty. Please wait for ebm.fit to complete.')

Training in progress or history empty. Please wait for ebm.fit to complete.


## 4. Generate images <a name="generate"></a>

In [ ]:
start_imgs = (
    np.random.uniform(size=(10, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)) * 2 - 1
)

In [ ]:
display(start_imgs)

In [ ]:
gen_img = generate_samples(
    ebm.model,
    start_imgs,
    steps=1000,
    step_size=STEP_SIZE,
    noise=NOISE,
    return_img_per_step=True,
)

In [ ]:
display(gen_img[-1].numpy())

In [ ]:
imgs = []
for i in [0, 1, 3, 5, 10, 30, 50, 100, 300, 999]:
    imgs.append(gen_img[i].numpy()[6])

display(np.array(imgs))